# MSCS 634 — Project Deliverable 2
## Regression Modeling and Performance Evaluation
**Student:** Shawn Miller  
**Course:** MSCS 634 — Data Mining  
**Dataset:** UCI Online Retail (cleaned dataset from Deliverable 1)

I want to see how well regression models estimate an invoice's total using information about the products purchased. I'll compare two Ridge models and a Random Forest, then check them against later invoices.

**Important limitation:** These models already know the products and quantities in an order. They estimate the order value, not a customer's spending before an order is placed.

## 1. Set up the notebook
I'll import the libraries and keep the same random seed each time.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
OUTPUT_DIR = Path('figures')
OUTPUT_DIR.mkdir(exist_ok=True)
pd.set_option('display.max_columns', 30)

## 2. Load the cleaned dataset
I'm using the cleaned sales data from Deliverable 1. I'll check the expected columns and types without changing the original file.

In [ ]:
DATA_PATH = Path('Online_Retail_Cleaned.csv.zip')
if not DATA_PATH.exists():
    raise FileNotFoundError(
        'Download Online_Retail_Cleaned.csv.zip from Deliverable 1 and place it next to this notebook.'
    )
with ZipFile(DATA_PATH) as archive:
    candidates = [f for f in archive.namelist() if f.lower().endswith('.csv')]
    if not candidates:
        raise FileNotFoundError('The ZIP archive contains no CSV file.')
    with archive.open(candidates[0]) as f:
        sales = pd.read_csv(f, low_memory=False)

sales.columns = sales.columns.str.strip()
required = {'InvoiceNo','StockCode','Quantity','InvoiceDate','UnitPrice','Country'}
missing = required - set(sales.columns)
if missing:
    raise ValueError(f'Missing expected columns: {sorted(missing)}')
sales['InvoiceNo'] = sales['InvoiceNo'].astype(str).str.strip()
sales['InvoiceDate'] = pd.to_datetime(sales['InvoiceDate'],errors='coerce')
for column in ['Quantity','UnitPrice']:
    sales[column] = pd.to_numeric(sales[column],errors='coerce')
sales = sales.dropna(subset=['InvoiceDate','Quantity','UnitPrice','InvoiceNo'])
sales = sales[(sales.Quantity > 0) & (sales.UnitPrice > 0) &
              ~sales.InvoiceNo.str.upper().str.startswith('C')].copy()
sales['line_sales'] = sales.Quantity * sales.UnitPrice
print('Valid sale lines:',f'{len(sales):,}')
print('Date range:',sales.InvoiceDate.min(),'to',sales.InvoiceDate.max())
display(sales.head(3))

## 3. Check unusually large orders
My instructor recommended looking more closely at extreme invoices. I'll flag the biggest ones and check how much revenue they represent instead of assuming they're errors.

In [ ]:
invoice_check = sales.groupby('InvoiceNo',as_index=False).agg(
    invoice_total=('line_sales','sum'),line_count=('StockCode','size'),
    units=('Quantity','sum'),max_line_sales=('line_sales','max'),
    date=('InvoiceDate','min'),country=('Country','first'))
threshold_99 = invoice_check.invoice_total.quantile(.99)
large = invoice_check[invoice_check.invoice_total > threshold_99]
share = 100 * large.invoice_total.sum() / invoice_check.invoice_total.sum()
print(f'99th percentile: £{threshold_99:,.2f}')
print(f'Invoices above threshold: {len(large):,}')
print(f'Revenue share from top 1%: {share:.2f}%')
display(invoice_check.nlargest(15,'invoice_total'))
print('These are review flags, not verified errors.')

## 4. Create invoice-level features
Each sales row describes a product line. I'll group rows into invoices and create features including quantities, product counts, prices, and purchase timing. I won't include the invoice total itself as a predictor.

In [ ]:
invoice = sales.groupby('InvoiceNo',as_index=False).agg(
    invoice_total=('line_sales','sum'),invoice_date=('InvoiceDate','min'),
    country=('Country','first'),n_lines=('StockCode','size'),
    unique_products=('StockCode','nunique'),total_units=('Quantity','sum'),
    avg_unit_price=('UnitPrice','mean'),max_unit_price=('UnitPrice','max'),
    min_unit_price=('UnitPrice','min'),avg_line_quantity=('Quantity','mean'))
invoice['month'] = invoice.invoice_date.dt.month
invoice['day_of_week'] = invoice.invoice_date.dt.dayofweek
invoice['hour'] = invoice.invoice_date.dt.hour
invoice['is_weekend'] = (invoice.day_of_week >= 5).astype(int)
invoice = invoice.sort_values(['invoice_date','InvoiceNo']).reset_index(drop=True)
assert invoice.InvoiceNo.is_unique
print('Invoices:',f'{len(invoice):,}')
display(invoice.invoice_total.describe(percentiles=[.5,.9,.95,.99]))

## 5. Train and test using different time periods
I'll use the earlier 80% of invoices for training and the latest 20% for testing. Each model will have the same final test set.

In [ ]:
cut = int(len(invoice)*.8)
train,test = invoice.iloc[:cut].copy(),invoice.iloc[cut:].copy()
base_features = ['n_lines','total_units','avg_unit_price']
engineered_features = base_features + [
    'unique_products','max_unit_price','min_unit_price','avg_line_quantity',
    'month','day_of_week','hour','is_weekend','country']
y_train,y_test = train.invoice_total,test.invoice_total
print(f'Training invoices: {len(train):,}; test invoices: {len(test):,}')
print('Train through:',train.invoice_date.max(),'| Test starts:',test.invoice_date.min())

## 6. Compare three regression models
I'll try basic Ridge, Ridge with extra features, and Random Forest. Pipelines make sure preprocessing is learned only from the training data.

In [ ]:
def make_preprocessor(features):
    numeric = [c for c in features if c != 'country']
    transforms = [('numeric',Pipeline([
        ('impute',SimpleImputer(strategy='median')),
        ('scale',StandardScaler())]),numeric)]
    if 'country' in features:
        transforms.append(('country',Pipeline([
            ('impute',SimpleImputer(strategy='most_frequent')),
            ('encode',OneHotEncoder(handle_unknown='ignore'))]),['country']))
    return ColumnTransformer(transforms)

models = {
 'Ridge (basic)':(base_features,Pipeline([
     ('prep',make_preprocessor(base_features)),('model',Ridge(alpha=10))])),
 'Ridge (engineered)':(engineered_features,Pipeline([
     ('prep',make_preprocessor(engineered_features)),('model',Ridge(alpha=10))])),
 'Random Forest (engineered)':(engineered_features,Pipeline([
     ('prep',make_preprocessor(engineered_features)),
     ('model',RandomForestRegressor(n_estimators=100,min_samples_leaf=3,
                                   max_features=.8,random_state=RANDOM_STATE,n_jobs=-1))]))
}
def metrics(y_true,pred):
    return {'MAE (£)':mean_absolute_error(y_true,pred),
            'MSE (£²)':mean_squared_error(y_true,pred),
            'RMSE (£)':np.sqrt(mean_squared_error(y_true,pred)),
            'R²':r2_score(y_true,pred)}
predictions={}
rows=[]
for name,(features,pipe) in models.items():
    pipe.fit(train[features],y_train)
    pred=pipe.predict(test[features])
    predictions[name]=pred
    rows.append({'Model':name,**metrics(y_test,pred)})
results=pd.DataFrame(rows).sort_values('RMSE (£)').reset_index(drop=True)
display(results.round(3))

## 7. Use cross-validation
I'll also run five rounds of time-series cross-validation on the training data. This helps show whether the results hold up across different periods.

In [ ]:
folds=TimeSeriesSplit(n_splits=5)
cv_rows=[]
for name,(features,pipe) in models.items():
    cv=cross_validate(pipe,train[features],y_train,cv=folds,
        scoring={'rmse':'neg_root_mean_squared_error','r2':'r2',
                 'mae':'neg_mean_absolute_error'},
        n_jobs=1,error_score='raise')
    cv_rows.append({'Model':name,'CV RMSE mean (£)':-cv['test_rmse'].mean(),
                    'CV RMSE std (£)':cv['test_rmse'].std(),
                    'CV MAE mean (£)':-cv['test_mae'].mean(),
                    'CV R² mean':cv['test_r2'].mean()})
cv_results=pd.DataFrame(cv_rows).sort_values('CV RMSE mean (£)')
display(cv_results.round(3))

## 8. Visualize the results
I'll compare RMSE and R², plot actual versus predicted invoice totals, and look at the prediction errors. A log scale makes smaller invoices easier to see beside the largest orders.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(13,4.5))
axes[0].bar(results['Model'],results['RMSE (£)'])
axes[0].set_title('Holdout RMSE by model')
axes[0].set_ylabel('RMSE (£; lower is better)')
axes[0].tick_params(axis='x',rotation=15)
axes[1].bar(results['Model'],results['R²'])
axes[1].set_title('Holdout R² by model')
axes[1].set_ylabel('R² (higher is better)')
axes[1].tick_params(axis='x',rotation=15)
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'model_metrics.png',dpi=150)
plt.show()

best_name=results.iloc[0]['Model']
best_pred=predictions[best_name]
fig,ax=plt.subplots(figsize=(6.5,5))
ax.scatter(y_test.clip(lower=1),np.maximum(best_pred,1),alpha=.25,s=8)
upper=max(y_test.max(),max(best_pred))
ax.plot([1,upper],[1,upper],linestyle='--',label='Perfect prediction')
ax.set(xscale='log',yscale='log',xlabel='Actual invoice value (£, log scale)',
       ylabel='Predicted invoice value (£, log scale)',
       title=f'Actual vs. predicted — {best_name}')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'actual_vs_predicted.png',dpi=150)
plt.show()

fig,ax=plt.subplots(figsize=(7,4))
ax.hist((y_test-best_pred).clip(lower=-1000,upper=1000),bins=55)
ax.set(xlabel='Prediction error (£; clipped to ±1,000 for display)',
       ylabel='Number of invoices',title='Distribution of prediction errors')
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'error_distribution.png',dpi=150)
plt.show()

## 9. Check sensitivity to extreme invoices
I'll compare errors on the full test set with errors on invoices below the training data's 99th percentile. This is a diagnostic, not a replacement for the full test scores.

In [ ]:
high_threshold=y_train.quantile(.99)
ordinary=(y_test<=high_threshold)
sensitivity=pd.DataFrame([
    {'Model':name,'All test RMSE (£)':metrics(y_test,pred)['RMSE (£)'],
     'Test RMSE below training 99th percentile (£)':metrics(y_test[ordinary],pred[ordinary])['RMSE (£)']}
    for name,pred in predictions.items()])
print(f'Training 99th-percentile invoice value: £{high_threshold:,.2f}')
print(f'Test invoices above threshold: {(~ordinary).sum():,} of {len(y_test):,}')
display(sensitivity.round(3))

## 10. Summarize what I learned
The code prints the actual results after the models run. I'll compare the newest test invoices with cross-validation and use the extreme-order check to explain any surprising differences.

The largest limitations are extreme orders, the short date range, and knowing the contents of each order before making a prediction. A better future-spending project would use only purchases made before the target period.

In [ ]:
winner=results.iloc[0]
base_rmse=float(results.loc[results.Model=='Ridge (basic)','RMSE (£)'].iloc[0])
eng_rmse=float(results.loc[results.Model=='Ridge (engineered)','RMSE (£)'].iloc[0])
cv_winner=cv_results.iloc[0]
print('RESULTS SUMMARY — Shawn Miller')
print(f"Best holdout model: {winner['Model']} (RMSE £{winner['RMSE (£)']:,.2f}, R² {winner['R²']:.3f}).")
print(f'Basic Ridge RMSE: £{base_rmse:,.2f}; engineered Ridge: £{eng_rmse:,.2f}.')
print(f'Engineered Ridge improvement in RMSE: £{base_rmse-eng_rmse:+,.2f} (positive is better).')
print(f"Lowest CV RMSE: {cv_winner['Model']} (£{cv_winner['CV RMSE mean (£)']:,.2f}).")
print('Unusual orders were flagged but not labeled errors without verification.')
results.to_csv('holdout_results.csv',index=False)
cv_results.to_csv('cross_validation_results.csv',index=False)
sensitivity.to_csv('outlier_sensitivity.csv',index=False)
print('Saved evaluation results and figures.')